In [ ]:
words = open('prenoms.txt', encoding='utf-8').read().splitlines()
words = [w for w in words if w.strip()]
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}

In [ ]:
import torch

block_size=3
X,Y=[],[]
for w in words:
    context=[0]*block_size
    for ch in w+'.' :
        ix=stoi[ch]
        X.append(context)
        Y.append(ix)
        context=context[1:]+[ix]

X=torch.tensor(X)
Y=torch.tensor(Y)

print(X.shape,Y.shape)

In [ ]:
embed_dim=10
C=torch.randn((46,embed_dim),requires_grad=True)

emb=C[X]
print(emb.shape)

In [ ]:
import torch.nn.functional as F

hidden_dim = 200
W1 = torch.randn((block_size * embed_dim, hidden_dim), requires_grad=True)   # (30, 200)
b1 = torch.randn(hidden_dim, requires_grad=True)
W2 = torch.randn((hidden_dim, 46), requires_grad=True)
b2 = torch.randn(46, requires_grad=True)
parameters = [C, W1, b1, W2, b2]
print(sum(p.nelement() for p in parameters))   # 15906

emb = C[X]                                        # (N, 3, 10)查表
h = torch.tanh(emb.view(-1, block_size * embed_dim) @ W1 + b1)   # (N, 200)
logits = h @ W2 + b2                              # (N, 46)
loss = F.cross_entropy(logits, Y)                 # 内部 = log_softmax + NLL，和上节课的 loss 等价


In [ ]:
from torch.utils.data import TensorDataset,DataLoader,random_split
dataset=TensorDataset(X,Y)

train_set,val_set,test_set=random_split(dataset,[0.8,0.1,0.1])
train_loader=DataLoader(train_set,batch_size=256,shuffle=True)
val_loader=DataLoader(val_set,batch_size=256,shuffle=False)
test_loader=DataLoader(test_set,batch_size=256,shuffle=False)

In [ ]:
import matplotlib.pyplot as plt
lre = torch.linspace(-3, 0, 1000)     # -3 ~ 0
lrs = 10 ** lre                        # 10⁻³ ~ 10⁰ = 0.001 ~ 1

lossi, lri = [], []
for i in range(1000):
    x, y = next(iter(train_loader))    # 每步取一个 batch
    emb = C[x]
    h = torch.tanh(emb.view(-1, 30) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, y)
    for p in parameters: p.grad = None
    loss.backward()
    for p in parameters:
        p.data -= lrs[i] * p.grad      # 每个 batch 用不同的 lr 更新一次
    lossi.append(loss.log10().item())
    lri.append(lre[i].item())

plt.plot(lri, lossi)                   # 曲线低谷对应的 lr 就是好选择

In [ ]:
lr, epochs = 0.2, 100
lossi, lossvali = [], []

for epoch in range(epochs):
    loss_epoch = 0.0
    for x, y in train_loader:
        emb = C[x]
        h = torch.tanh(emb.view(-1, 30) @ W1 + b1)
        logits = h @ W2 + b2
        loss = F.cross_entropy(logits, y)
        for p in parameters: p.grad = None
        loss.backward()
        cur_lr = lr if epoch < 50 else lr * 0.1
        for p in parameters:
            p.data -= cur_lr * p.grad
        loss_epoch += loss.item()
    loss_epoch /= len(train_loader)

    loss_val = 0.0
    for x, y in val_loader:
        emb = C[x]
        h = torch.tanh(emb.view(-1, 30) @ W1 + b1)
        logits = h @ W2 + b2
        loss_val += F.cross_entropy(logits, y).item()
    loss_val /= len(val_loader)

    lossi.append(loss_epoch)
    lossvali.append(loss_val)
    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d} | train {loss_epoch:.3f} | val {loss_val:.3f}')

# 训练结束后画曲线
import matplotlib.pyplot as plt
plt.plot(lossi, label='train')
plt.plot(lossvali, label='val')
plt.xlabel('epoch'); plt.ylabel('loss'); plt.legend(); plt.show()

In [ ]:
def gen_name():
    context = [0] * block_size
    out = []
    while True:
        emb = C[torch.tensor([context])]
        h = torch.tanh(emb.view(1, -1) @ W1 + b1)
        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1).item()
        context = context[1:] + [ix]
        out.append(itos[ix])
        if ix == 0: break
    return ''.join(out)

for _ in range(20):
    print(gen_name())

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(C[:, 0].data, C[:, 1].data, s=200)
for i in range(C.shape[0]):
    plt.text(C[i, 0].item(), C[i, 1].item(), itos[i], ha='center', va='center')
plt.show()

In [8]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader,random_split
import torch.nn.functional as F

words=open("prenoms.txt",encoding='utf-8').read().splitlines()
words=[w for w in words]
chars=sorted(list(set(''.join(words))))
stoi={s:i for i,s in enumerate(chars)}
stoi['.']=0
itos={i:s for s,i in stoi.items()}
V=len(stoi)
X,Y=[],[]
block_size=3

for w in words:
    context=[0]*block_size
    for ch in w+'.':
        ix=stoi[ch]
        X.append(context)
        Y.append(ix)
        context=context[1:]+[ix]

X=torch.tensor(X)
Y=torch.tensor(Y)

n=len(X)
dataset=TensorDataset(X,Y)

train_set,val_set,test_set=random_split(dataset,[int(0.8*n),int(0.1*n),n-int(0.8*n)-int(0.1*n)])

train_loader=DataLoader(train_set,batch_size=256,shuffle=True)
val_loader=DataLoader(val_set,batch_size=256,shuffle=False)
test_loader=DataLoader(test_set,batch_size=256,shuffle=False)

class CharLM(nn.Module):
    def __init__(self, vocal_size,block_size,embed_dim,hidden_dim):
        super().__init__()
        self.embedding=nn.Embedding(vocal_size,embed_dim)
        self.net=nn.Sequential(
            nn.Linear(block_size*embed_dim,hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim,vocal_size),
        )
    
    def forward(self,x):
        emb=self.embedding(x)
        emb=emb.view(x.size(0),-1)
        return self.net(emb)
    

model=CharLM(V,block_size,embed_dim=10,hidden_dim=200)

print(f"参数量：{sum(p.numel() for p in model.parameters())}")






参数量：15906


In [9]:
opt=torch.optim.SGD(model.parameters(),lr=0.2)

epochs=100
for k in range(epochs):
    model.train()
    loss_train=0.0

    for x,y in train_loader:
        preds=model(x)
        loss=F.cross_entropy(preds,y)
        loss_train+=loss.item()
        opt.zero_grad()
        loss.backward()
        opt.step()

    model.eval()
    loss_val=0.0

    with torch.no_grad():
       for x,y in val_loader:
          preds=model(x)
          loss=F.cross_entropy(preds,y)
          loss_val+=loss.item()

    if(k==0 or (k+1)%10==0):
      print(f"{k+1}: loss_train={loss_train/len(train_loader):.3f},loss_val={loss_val/len(val_loader):.3f}")


1: loss_train=2.576,loss_val=2.482
10: loss_train=2.185,loss_val=2.240
20: loss_train=2.119,loss_val=2.193
30: loss_train=2.091,loss_val=2.184
40: loss_train=2.075,loss_val=2.223
50: loss_train=2.063,loss_val=2.171
60: loss_train=2.057,loss_val=2.151
70: loss_train=2.050,loss_val=2.151
80: loss_train=2.046,loss_val=2.184
90: loss_train=2.041,loss_val=2.145
100: loss_train=2.039,loss_val=2.155


In [10]:
model.eval()
def gen_name():
    context = [0] * block_size
    out = []
    with torch.no_grad():
        while True:
            logits = model(torch.tensor([context]))
            probs = F.softmax(logits, dim=1)
            ix = torch.multinomial(probs, 1).item()
            context = context[1:] + [ix]
            out.append(itos[ix])
            if ix == 0: break
    return ''.join(out)

for _ in range(10):
    print(gen_name())

MON.
BRAH.
MARLAN.
WILEM.
DJEMMANETTE.
BONAR.
BELLONE.
ARMA.
ARTYS.
LOCION-PYRÈMETHIANA.
